In [1]:
# Before everything:
!pip uninstall -y transformers trl peft numpy pandas
!pip install -q numpy==1.26.4  # This is the critical line!

Found existing installation: transformers 4.57.1
Uninstalling transformers-4.57.1:
  Successfully uninstalled transformers-4.57.1
Found existing installation: peft 0.17.1
Uninstalling peft-0.17.1:
  Successfully uninstalled peft-0.17.1
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 95.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.4.1 requires pandas, which is not installed.
woodwork 0.31.0 requires pandas>=2.0.0, which is not installed.
kauldron 1.3.0 requires pandas, which is not installed.
featuretools 1.31.0 requires

In [6]:
# ==============================================================
# 🚀 DPO Performance Master Benchmarker (v3 FINAL)
# Optimized for Kaggle 2×T4 GPUs
# Methods: Standard, FSDP, ZeRO-2, ZeRO-2-Optim, ZeRO-3
# ==============================================================

import os
import json
import time
import subprocess

# 1. THE UNIVERSAL WORKER SCRIPT
training_script = '''
import os
import torch
import json
import time
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig

# Patch for stability
import transformers.modeling_utils
def robust_patch(func=None):
    if func is None: return robust_patch
    return func
transformers.modeling_utils.check_model_inputs = robust_patch

MODE = os.environ.get("BENCHMARK_MODE", "standard")
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
DATASET_NAME = "Anthropic/hh-rlhf"

# Dataset Prep (1000 samples)
def extract_dpo_data(example):
    if "Assistant:" not in example.get("chosen", ""): return None
    try:
        p_chosen = example["chosen"].rsplit("Assistant:", 1)
        p_rejected = example["rejected"].rsplit("Assistant:", 1)
        return {"prompt": p_chosen[0] + "Assistant:", "chosen": p_chosen[1].lstrip(), "rejected": p_rejected[1].lstrip()}
    except: return None

dataset = load_dataset(DATASET_NAME, split="train[:1000]")
dataset = dataset.map(extract_dpo_data).filter(lambda x: x is not None)
dataset = dataset.train_test_split(test_size=0.1)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 1. Setup Training Args FIRST (Mandatory for Zero-3)
train_cfg = {
    "output_dir": "./temp_out",
    "per_device_train_batch_size": 1 if MODE == "zero3" else 2,
    "gradient_accumulation_steps": 8 if MODE == "zero3" else 4,
    "learning_rate": 5e-7,
    "num_train_epochs": 1,
    "beta": 0.1,
    "max_length": 512,
    "max_prompt_length": 256,
    "fp16": True,
    "gradient_checkpointing": True if MODE != "zero3" else False,
    "logging_steps": 10,
    "save_strategy": "no",
    "report_to": "none",
    "remove_unused_columns": False
}

if MODE == "fsdp":
    train_cfg["fsdp"] = "full_shard auto_wrap"
    train_cfg["fsdp_config"] = {"fsdp_transformer_layer_cls_to_wrap": ["LlamaDecoderLayer"]}
elif "zero" in MODE:
    train_cfg["deepspeed"] = f"/kaggle/working/ds_config_{MODE.replace('_sexy','')}.json"

args = DPOConfig(**train_cfg)

# 2. Load Model (After args for Zero-3 compatibility)
device_map = "auto" if MODE == "standard" else None
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16, device_map=device_map, attn_implementation="sdpa"
)
model.config.use_cache = False

peft_config = LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM")

# Fix for AttributeError from previous run
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

# 3. Trainer & Benchmarking
trainer = DPOTrainer(
    model=model, ref_model=None, args=args, train_dataset=dataset["train"], 
    eval_dataset=dataset["test"], tokenizer=tokenizer, peft_config=peft_config
)

start = time.time()
trainer.train()
end = time.time()

if int(os.environ.get("RANK", 0)) == 0:
    peak_gpu = torch.cuda.max_memory_allocated() / (1024**3)
    duration = end - start
    with open(f"/kaggle/working/res_{MODE}.json", "w") as f:
        json.dump({"M": MODE.upper(), "GPU": round(peak_gpu, 2), "T": round(duration, 2), "S": round(len(dataset["train"])/duration, 2)}, f)
'''

# 2. UTILITIES
def setup_files():
    with open('/kaggle/working/bench_run.py', 'w') as f: f.write(training_script)
    ds_base = {"fp16": {"enabled": True}, "gradient_accumulation_steps": "auto", "train_batch_size": "auto", "train_micro_batch_size_per_gpu": "auto"}
    
    # Zero configs
    z2 = {**ds_base, "zero_optimization": {"stage": 2, "offload_optimizer": {"device": "cpu"}}}
    z2s = {**ds_base, "zero_optimization": {"stage": 2, "offload_optimizer": {"device": "cpu"}, "allgather_bucket_size": 2e8, "reduce_bucket_size": 2e8}}
    z3 = {**ds_base, "zero_optimization": {"stage": 3, "offload_optimizer": {"device": "cpu"}, "offload_param": {"device": "cpu"}}}
    
    for n, c in [("zero2", z2), ("zero2_sexy", z2s), ("zero3", z3)]:
        with open(f"/kaggle/working/ds_config_{n}.json", "w") as f: json.dump(c, f)

def run():
    tests = [
        {"m": "standard", "c": ["python", "/kaggle/working/bench_run.py"]},
        {"m": "fsdp", "c": ["accelerate", "launch", "--multi_gpu", "--num_processes", "2", "/kaggle/working/bench_run.py"]},
        {"m": "zero2", "c": ["deepspeed", "--num_gpus=2", "/kaggle/working/bench_run.py"]},
        {"m": "zero2_sexy", "c": ["deepspeed", "--num_gpus=2", "/kaggle/working/bench_run.py"]},
        {"m": "zero3", "c": ["deepspeed", "--num_gpus=2", "/kaggle/working/bench_run.py"]},
    ]
    for t in tests:
        print(f"\n⚡ TESTING: {t['m'].upper()}...")
        e = os.environ.copy(); e["BENCHMARK_MODE"] = t["m"]
        subprocess.run(t["c"], env=e)

def table():
    print("\n" + "="*75)
    print(f"{'METHOD':<20} | {'GPU (GB)':<12} | {'TIME (S)':<12} | {'SAMPLES/S'}")
    print("-" * 75)
    for m in ["standard", "fsdp", "zero2", "zero2_sexy", "zero3"]:
        p = f"/kaggle/working/res_{m}.json"
        if os.path.exists(p):
            with open(p, "r") as f:
                r = json.load(f)
                print(f"{r['M'].replace('_SEXY','_OPTIM'):<20} | {r['GPU']:<12.2f} | {r['T']:<12.2f} | {r['S']:<12.2f}")
    print("="*75)

# --- EXEC ---
print("Initializing Benchmark...")
setup_files()
run()
table()

Initializing Benchmark...

⚡ TESTING: STANDARD...


2025-12-20 23:58:34.300684: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766275114.322013    1356 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766275114.328601    1356 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766275114.345532    1356 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766275114.345555    1356 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766275114.345559    1356 computation_placer.cc:177] computation placer alr

{'loss': 0.6929, 'grad_norm': 3.9980714321136475, 'learning_rate': 4.5535714285714285e-07, 'rewards/chosen': 7.667540921829641e-05, 'rewards/rejected': -0.00048015592619776726, 'rewards/accuracies': 0.550000011920929, 'rewards/margins': 0.0005568313645198941, 'logps/rejected': -106.6649398803711, 'logps/chosen': -95.69422149658203, 'logits/rejected': -2.826874256134033, 'logits/chosen': -2.74766206741333, 'epoch': 0.09}
{'loss': 0.6929, 'grad_norm': 4.395798206329346, 'learning_rate': 4.1517857142857145e-07, 'rewards/chosen': 0.0002237940498162061, 'rewards/rejected': -0.00029506205464713275, 'rewards/accuracies': 0.550000011920929, 'rewards/margins': 0.0005188560462556779, 'logps/rejected': -132.33702087402344, 'logps/chosen': -111.51979064941406, 'logits/rejected': -2.8918426036834717, 'logits/chosen': -2.8168728351593018, 'epoch': 0.18}
{'loss': 0.6929, 'grad_norm': 3.6266520023345947, 'learning_rate': 3.7053571428571427e-07, 'rewards/chosen': 0.0006720948731526732, 'rewards/rejecte

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

{'loss': 0.6928, 'grad_norm': 2.931195020675659, 'learning_rate': 4.1964285714285714e-07, 'rewards/chosen': 0.00047851563431322575, 'rewards/rejected': -0.00026239396538585424, 'rewards/accuracies': 0.612500011920929, 'rewards/margins': 0.00074090959969908, 'logps/rejected': -122.35096740722656, 'logps/chosen': -110.49405670166016, 'logits/rejected': -2.903414249420166, 'logits/chosen': -2.816929340362549, 'epoch': 0.18}


...

{'loss': 0.6928, 'grad_norm': 2.566971778869629, 'learning_rate': 1.6071428571428573e-07, 'rewards/chosen': -0.00012735367636196315, 'rewards/rejected': -0.0006607341929338872, 'rewards/accuracies': 0.5625, 'rewards/margins': 0.0005333805456757545, 'logps/rejected': -99.45231628417969, 'logps/chosen': -82.92845916748047, 'logits/rejected': -3.222156047821045, 'logits/chosen': -3.0855910778045654, 'epoch': 0.71}


 89%|████████▉ | 50/56 [16:10<01:56, 19.38s/it]

{'loss': 0.6931, 'grad_norm': 2.99247407913208, 'learning_rate': 7.142857142857142e-08, 'rewards/chosen': 8.943556167650968e-05, 'rewards/rejected': -0.000145585581776686, 'rewards/accuracies': 0.5, 'rewards/margins': 0.0002350211434531957, 'logps/rejected': -118.66334533691406, 'logps/chosen': -102.84210205078125, 'logits/rejected': -3.2779197692871094, 'logits/chosen': -3.1855628490448, 'epoch': 0.89}


100%|██████████| 56/56 [18:06<00:00, 19.39s/it]


{'train_runtime': 1086.0933, 'train_samples_per_second': 0.829, 'train_steps_per_second': 0.052, 'train_loss': 0.693031200340816, 'epoch': 1.0}
[2025-12-21 00:41:31,013] [INFO] [launch.py:367:main] Process 2405 exits successfully.
[2025-12-21 00:41:32,013] [INFO] [launch.py:367:main] Process 2404 exits successfully.

METHOD               | GPU (GB)     | TIME (S)     | SAMPLES/S
---------------------------------------------------------------------------
STANDARD             | 2.48         | 336.79       | 2.67        
FSDP                 | 2.91         | 517.47       | 1.74        
ZERO2                | 3.62         | 242.60       | 3.71        
ZERO2_OPTIM          | 3.62         | 208.76       | 4.31        
ZERO3                | 4.20         | 1097.17      | 0.82        
